In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 68.3 MB/s eta 0:00:00


In [5]:
import pandas as pd
import ast
import math
import warnings

warnings.filterwarnings("ignore")

# Read the CSV using the python engine for more robust parsing
# Set on_bad_lines to 'warn' to handle unexpected end of data or other malformed lines gracefully
df_raw = pd.read_csv("IMDB_prep.csv", engine='python', on_bad_lines='warn')

# Replace string 'nan' with an empty list representation '[]'
# This handles cases where 'nan' is present as a string and causes issues with ast.literal_eval
df_raw['tokenized'] = df_raw['tokenized'].replace('nan', '[]')

# Now apply ast.literal_eval to convert the string representations of lists to actual lists
df = df_raw["tokenized"].apply(ast.literal_eval).tolist();

df

[['one',
  'the',
  'other',
  'reviewers',
  'has',
  'mentioned',
  'that',
  'after',
  'watching',
  'just',
  'episode',
  'you',
  'hooked',
  'they',
  'are',
  'right',
  'this',
  'exactly',
  'what',
  'happened',
  'with',
  'the',
  'first',
  'thing',
  'that',
  'struck',
  'about',
  'was',
  'its',
  'brutality',
  'and',
  'unflinching',
  'scenes',
  'violence',
  'which',
  'set',
  'right',
  'from',
  'the',
  'word',
  'trust',
  'this',
  'not',
  'show',
  'for',
  'the',
  'faint',
  'hearted',
  'timid',
  'this',
  'show',
  'pulls',
  'punches',
  'with',
  'regards',
  'drugs',
  'sex',
  'violence',
  'its',
  'hardcore',
  'the',
  'classic',
  'use',
  'the',
  'word',
  'called',
  'that',
  'the',
  'nickname',
  'given',
  'the',
  'oswald',
  'maximum',
  'security',
  'state',
  'focuses',
  'mainly',
  'emerald',
  'city',
  'experimental',
  'section',
  'the',
  'prison',
  'where',
  'all',
  'the',
  'cells',
  'have',
  'glass',
  'fronts',
  

In [6]:
from gensim.models import FastText

In [7]:
# FastText modeli eğit
ft_model = FastText(
    sentences=df,
    vector_size=100,
    window=3,
    min_count=2,
    sg=1,           # Skip-gram
    epochs=20,
    seed=42
)

print(f"FastText modeli eğitildi.")
print(f"Kelime dağarcığı: {len(ft_model.wv)}")
print(f"Vektör boyutu: {ft_model.wv.vector_size}")

FastText modeli eğitildi.
Kelime dağarcığı: 6126
Vektör boyutu: 100


In [8]:
for kelime in ["film", "movie", "action"]:
    print(f"'{kelime}' -> en benzer 3 kelime:")
    try:
        for k, s in ft_model.wv.most_similar(kelime, topn=3):
            print(f"    {k} ({s:.4f})")
    except:
        print(f"    (bulunamadı)")
    print()

'film' -> en benzer 3 kelime:
    file (0.8123)
    filmed (0.7942)
    films (0.7916)

'movie' -> en benzer 3 kelime:
    movies (0.8455)
    move (0.7924)
    pie (0.7691)

'action' -> en benzer 3 kelime:
    fraction (0.9407)
    function (0.9271)
    section (0.9250)



In [9]:
# OOV test: Out Of Vocavulary
oov_kelime = "prehistoric"  # eğitimde yok
varsa = ft_model.wv.has_index_for(oov_kelime)
print(f"\nOOV Test: '{oov_kelime}' eğitimde var mı? {varsa}")
try:
    vektor = ft_model.wv[oov_kelime]
    print(f"  Ama FastText yine de vektör üretti (subword'ler sayesinde)!")
    print(f"  Vektör (ilk 5 değer): {vektor[:5]}")
except:
    print(f"  Vektör üretilemedi (küçük korpus, yetersiz subword)")


OOV Test: 'prehistoric' eğitimde var mı? False
  Ama FastText yine de vektör üretti (subword'ler sayesinde)!
  Vektör (ilk 5 değer): [ 0.03975208 -0.07775656  0.09691246  0.0873099  -0.1234142 ]
